# 🤝 Mental Health Continuity & Handover System - Interactive Notebook

This notebook provides a complete interactive walkthrough of the **Consent-Aware Mental Health Continuity Summary System**.

- **Semantic NLP Redaction**: Sentence-level paraphrase topic classification with provenance tagging.
- **Zero-Hallucination Integrity**: Mathematical verification that all assertions correlate to source notes.
- **Emergency Break-Glass Override (IR-04)**: Audited crisis overrides in SQLite.
- **Longitudinal Multi-Clinician EHR**: Multi-month handovers across psychiatrists, nurses, counsellors, and social workers.
- **Quantitative Benchmark**: Baseline vs. Prototype metric evaluation across 9 clients.

In [ ]:
import sys
import os
from pathlib import Path

# Robust sys.path configuration for local CLI or Jupyter execution
cwd = Path(os.getcwd())
if (cwd / 'src').exists():
    sys.path.insert(0, str(cwd))
elif (cwd.parent / 'src').exists():
    sys.path.insert(0, str(cwd.parent))

from src.db.database import DatabaseManager
from src.models.data_models import RoleEnum
from src.engine.summary_engine import SummaryEngine
from src.engine.nlp_consent_engine import ProvenanceTracker
from src.engine.emergency_override import EmergencyOverrideRequest
from src.evaluation.benchmark import print_benchmark_report

# 1. Initialize SQLite Database (runs migrations & auto-seeds)
db = DatabaseManager()
print('✅ SQLite Database Initialized, Migrated, and Seeded!')
print('Available Client IDs:', db.list_client_ids())

In [ ]:
# 2. Load Client Profile (Alex Rivera - MH-101)
client = db.get_client_profile('MH-101')
engine = SummaryEngine(client)

print('=== [1] TRADITIONAL BASELINE (RAW UN-REDACTED DUMP) ===')
print(engine.generate_baseline_summary(RoleEnum.SOCIAL_WORKER))

print('\n=== [2] PROTOTYPE (CONSENT-AWARE CONTINUITY HANDOVER) ===')
summary = engine.generate_prototype_summary(RoleEnum.SOCIAL_WORKER)
print(summary.disclaimer_header)
print(f'\nClient: {summary.client_name} ({summary.client_id}) | Target Role: {summary.target_role.value.upper()}')
print('\nConsented Operational Notes:')
for note in summary.operational_support_notes:
    print('  •', note)
print('\nRedaction Notices:')
for red in summary.redacted_sensitive_topics:
    print('  🚫', red)
print('\nActive Goals Preserved:')
for g in summary.active_goals:
    print(f'  🎯 [{g.priority.upper()}] {g.title} ({g.category}) - Status: {g.status}')
print('\nPending Actions:')
for a in summary.pending_actions:
    print(f'  📌 Due {a.due_date}: {a.description}')

In [ ]:
# 3. Longitudinal Multi-Clinician Cross-Handover (Marcus Vance - MH-202)
marcus = db.get_client_profile('MH-202')
marcus_engine = SummaryEngine(marcus)
marcus_sw = marcus_engine.generate_prototype_summary(RoleEnum.SOCIAL_WORKER)
print(f'=== Longitudinal Handover for Veteran {marcus.preferred_name} ===')
print(f'Notes count across care team: {len(marcus.session_notes)}')
print('\nConsented Notes for Social Worker (VA Housing & Reintegration):')
for n in marcus_sw.operational_support_notes:
    print('  •', n)
print('\nScrubbed Notes (Combat Trauma & Alcohol Withdrawal Protected):')
for r in marcus_sw.redacted_sensitive_topics:
    print('  🚫', r)

In [ ]:
# 4. Execute Crisis Safety Break-Glass Override (IR-04)
em_req = EmergencyOverrideRequest(
    client_id='MH-101',
    duty_clinician_id='Nurse Kelly, RN',
    emergency_reason='Acute suicidal crisis de-escalation required',
    supervisor_approval_code='SUPERVISOR-EMERGENCY-AUTH'
)
em_summary = engine.generate_prototype_summary(RoleEnum.DUTY_TEAM, emergency_request=em_req)
print(em_summary.disclaimer_header)
print('\nBreak-Glass Audit Logs in SQLite:')
logs = db.get_emergency_audit_logs('MH-101')
for l in logs:
    print(f"  Audit #{l['log_id']} | Clinician: {l['duty_clinician_id']} | Reason: {l['emergency_reason']}")

In [ ]:
# 5. Zero-Hallucination Provenance Verification
provenance_check = ProvenanceTracker.verify_provenance_integrity(
    summary.operational_support_notes, client.session_notes
)
print('=== Provenance & Hallucination Control Verification ===')
for k, v in provenance_check.items():
    print(f'  {k}: {v}')

In [ ]:
# 6. Execute Quantitative Benchmark Suite
print_benchmark_report()